# Catálogo Corporativo de Metadados — Camada Bronze (Credit Risk)

Este notebook implementa um **catálogo corporativo de metadados** para as tabelas Bronze do catálogo `credit_risk`, seguindo boas práticas de Engenharia de Dados corporativa com Databricks, Unity Catalog, Delta Lake e Arquitetura Medalhão.

### Escopo
- **Metadados Técnicos** (16 campos): catalog_name, schema_name, ordinal_position, data_type, nullable, precision, scale, max_length, partition_column, column_comment, etc.
- **Metadados de Governança** (22 campos): description, business_definition, domain, subdomain, data_owner, data_steward, source_system, data_classification, sensitivity_level, lgpd_flag, pii_flag, financial_data_flag, customer_data_flag, critical_data_element, etc.
- **Metadados de Qualidade** (10 campos): null_count, null_percentage, distinct_count, data_quality_score, quality_status
- **Metadados Delta Lake** (4 campos): delta_version, last_commit_timestamp, last_operation, partition_columns
- **Estatísticas de Tabela** (8 campos): row_count, column_count, file_count, table_size_bytes, table_size_mb, avg_record_size
- **Data Lineage** (10 campos): source_catalog, source_table, source_file, target_catalog, target_table, transformation_type
- **Metadados ML** (8 campos): feature_name, feature_type, feature_group, target_variable_flag, model_consumption_flag
- **Metadados de Auditoria** (16 campos): ingestion_timestamp, created_at, notebook_name, user_name, execution_status, etc.

### Entregas
- 3 tabelas Delta persistidas em `credit_risk.bronze`:
  - `metadata_catalog_columns` — 355 registros, 73 campos
  - `metadata_catalog_tables` — 8 registros, 32 campos
  - `metadata_catalog_lineage` — 8 registros, 10 campos
- Campos automáticos extraídos de: Spark, Unity Catalog, Delta Lake, Databricks
- Campos manuais preparados como estrutura para preenchimento pela governança de negócio

In [0]:
# Lista as tabelas existentes no schema Bronze
spark.sql("""
    SHOW TABLES IN credit_risk.bronze
""").show(truncate=False)

In [0]:
# Importa funções e tipos necessários do PySpark
from pyspark.sql import Row
from pyspark.sql.types import StringType, BooleanType, ArrayType

In [0]:
# Obtém todas as tabelas existentes no schema Bronze
tabelas = (
    spark.sql("SHOW TABLES IN credit_risk.bronze")
    .filter("isTemporary = false")
    .select("tableName")
    .collect()
)

# Converte o resultado para uma lista de nomes
tabelas = [row["tableName"] for row in tabelas]

# Exclui tabelas de metadados/sistema
tabelas = [t for t in tabelas if t not in ("bronze_metadata", "metadata_catalog")]

# Exibe as tabelas encontradas
tabelas

## 1. Extração de Metadados Técnicos

Percorre cada tabela Bronze e extrai: `table_name`, `column_name`, `data_type`, `nullable`

In [0]:
# Lista para armazenar os metadados técnicos extraídos
metadados = []

# Percorre cada tabela da camada Bronze
for tabela in tabelas:

    # Obtém o schema da tabela
    df = spark.table(f"credit_risk.bronze.{tabela}")

    # Percorre todas as colunas do schema
    for campo in df.schema.fields:

        metadados.append(
            Row(
                table_name=tabela,
                column_name=campo.name,
                data_type=campo.dataType.simpleString(),
                nullable=campo.nullable
            )
        )

# Cria um DataFrame Spark com os metadados técnicos
df_metadados_tecnicos = spark.createDataFrame(metadados)

# Exibe o total de colunas catalogadas
print(f"Total de colunas extraídas: {df_metadados_tecnicos.count()}")
display(df_metadados_tecnicos.limit(20))

## 2. Dicionário de Metadados de Negócio

Extrai descrições oficiais do arquivo `HomeCredit_columns_description.csv` para todas as colunas e tabelas.
Inclui descrições a nível de tabela (governança de dados) e dicionário de negócio em português para as principais colunas.

In [0]:
# ============================================================================
# 1. Leitura das descrições oficiais do HomeCredit_columns_description.csv
# ============================================================================
# Este arquivo contém descrições em inglês para todas as colunas do dataset
df_desc_csv = (
    spark.read
    .option("encoding", "UTF-8")
    .csv(
        "/Volumes/credit_risk/bronze/volume/home-credit-default-risk/HomeCredit_columns_description.csv",
        header=True,
        escape='"'
    )
)

# O CSV possui uma coluna de índice não nomeada (_c0) antes das colunas reais
# Colunas reais: _c0, Table, Row, Description, Special
df_desc_csv = df_desc_csv.select("Table", "Row", "Description")

# Mapeia nomes de tabela do CSV para nossos nomes de tabela Bronze
# O CSV usa "application_{train|test}.csv" que corresponde a ambas as tabelas
csv_descriptions = {}
for row in df_desc_csv.collect():
    table_raw = row["Table"]
    column_name = str(row["Row"]).strip() if row["Row"] else ""
    description = str(row["Description"]).strip() if row["Description"] else ""

    if table_raw and "application" in str(table_raw):
        csv_descriptions[("application_train", column_name)] = description
        csv_descriptions[("application_test", column_name)] = description
    elif table_raw and "POS_CASH" in str(table_raw):
        csv_descriptions[("pos_cash_balance", column_name)] = description
    elif table_raw:
        table_name = str(table_raw).replace(".csv", "").lower()
        csv_descriptions[(table_name, column_name)] = description

print(f"Descrições oficiais carregadas: {len(csv_descriptions)} entradas do CSV")
# Debug: mostra amostra das descrições carregadas
for i, (k, v) in enumerate(csv_descriptions.items()):
    if i < 3:
        print(f"  {k}: {v[:70]}...")

# ============================================================================
# 2. Descrições a nível de tabela (governança de dados)
# ============================================================================
table_descriptions = {
    "application_train": "Dados de solicitação de empréstimo para clientes com histórico de pagamento (inclui variável TARGET)",
    "application_test": "Dados de solicitação de empréstimo para clientes sem histórico de pagamento (sem variável TARGET)",
    "bureau": "Histórico de crédito do cliente em bureaus de crédito externos",
    "bureau_balance": "Saldo mensal de créditos anteriores reportados ao bureau de crédito",
    "credit_card_balance": "Saldo mensal de cartões de crédito anteriores do cliente",
    "installments_payments": "Histórico de pagamentos de parcelas de empréstimos anteriores",
    "pos_cash_balance": "Saldo mensal de empréstimos anteriores em lojas (POS - Point of Sale) e empréstimos em dinheiro",
    "previous_application": "Solicitações de empréstimo anteriores do cliente na mesma instituição"
}

# ============================================================================
# 3. Dicionário de metadados de negócio (descrições aprimoradas em português)
# ============================================================================
# Prioridade: business_metadata > csv_descriptions > descrição gerada automaticamente
business_metadata = {
    "SK_ID_CURR": {
        "description": "Identificador único do empréstimo/cliente na tabela application_train",
        "domain": "Credit Risk",
        "data_classification": "Confidential",
        "tags": ["pii", "customer", "identifier"]
    },
    "TARGET": {
        "description": "Variável alvo binária: 1 = cliente teve dificuldade de pagamento (inadimplência), 0 = empréstimo reembolsado",
        "domain": "Credit Risk",
        "data_classification": "Confidential",
        "tags": ["risk", "ml_feature", "target"]
    },
    "AMT_INCOME_TOTAL": {
        "description": "Renda total declarada pelo cliente no momento da solicitação do empréstimo",
        "domain": "Credit Risk",
        "data_classification": "Confidential",
        "tags": ["financial", "customer", "ml_feature"]
    },
    "AMT_CREDIT": {
        "description": "Valor total do crédito (empréstimo) solicitado pelo cliente",
        "domain": "Credit Risk",
        "data_classification": "Confidential",
        "tags": ["financial", "ml_feature"]
    },
    "AMT_ANNUITY": {
        "description": "Valor da anuidade (parcela periódica) do empréstimo",
        "domain": "Credit Risk",
        "data_classification": "Confidential",
        "tags": ["financial", "ml_feature"]
    },
    "CODE_GENDER": {
        "description": "Gênero do cliente (M = Masculino, F = Feminino, XNA = Não informado)",
        "domain": "Credit Risk",
        "data_classification": "Confidential",
        "tags": ["pii", "customer", "demographic"]
    },
    "DAYS_BIRTH": {
        "description": "Idade do cliente em dias no momento da solicitação (valor negativo relativo ao início do empréstimo)",
        "domain": "Credit Risk",
        "data_classification": "Confidential",
        "tags": ["pii", "customer", "demographic", "ml_feature"]
    },
    "DAYS_EMPLOYED": {
        "description": "Tempo de emprego do cliente em dias no momento da solicitação (valor negativo)",
        "domain": "Credit Risk",
        "data_classification": "Confidential",
        "tags": ["financial", "customer", "ml_feature"]
    }
}

print(f"Dicionário de negócio definido com {len(business_metadata)} colunas aprimoradas em português")
print(f"Descrições de tabela definidas para {len(table_descriptions)} tabelas")

In [0]:
# Regras de classificação automática para colunas não documentadas no dicionário
# Padrões aplicados:
#   - Colunas financeiras (AMT_, etc.)          → Confidential + tag "financial"
#   - Colunas PII/demográficas (SK_ID_, etc.)   → Confidential + tags ["pii", "customer"]
#   - Colunas de risco (TARGET, etc.)            → Confidential + tag "risk"
#   - Colunas técnicas (FLAG_, etc.)             → Internal + tag "technical"
#   - Padrão: Confidential + tag "ml_feature" (máxima cautela)

# Prefixos e padrões para classificação automática
FINANCIAL_PATTERNS = ["AMT_", "CNT_", "PAYMENT", "CREDIT", "DEBT", "INCOME", "ANNUITY"]
PII_PATTERNS = ["SK_ID", "CODE_GENDER", "DAYS_BIRTH", "NAME_", "ORGANIZATION", "EMAIL", "PHONE"]
RISK_PATTERNS = ["TARGET", "DEFAULT", "RISK", "SCORE"]
TECHNICAL_PATTERNS = ["FLAG_", "_MODE", "WEEKDAY", "HOUR_"]


def classify_column(column_name):
    """Classifica uma coluna automaticamente com base no nome.

    Retorna: (data_classification, tags)
    """
    col_upper = column_name.upper()

    # Verifica se é PII / demográfica
    if any(col_upper.startswith(p) or p in col_upper for p in PII_PATTERNS):
        return "Confidential", ["pii", "customer"]

    # Verifica se é financeira
    if any(col_upper.startswith(p) for p in FINANCIAL_PATTERNS):
        return "Confidential", ["financial"]

    # Verifica se é de risco
    if any(p in col_upper for p in RISK_PATTERNS):
        return "Confidential", ["risk"]

    # Verifica se é técnica
    if any(col_upper.startswith(p) for p in TECHNICAL_PATTERNS):
        return "Internal", ["technical"]

    # Padrão: Confidential (máxima cautela)
    return "Confidential", ["ml_feature"]


def generate_description(column_name):
    """Gera uma descrição automática a partir do nome da coluna em snake_case."""
    words = column_name.replace("_", " ").lower().split()
    readable = " ".join(words)
    return f"Coluna {column_name} — {readable}"


# Constrói o catálogo completo combinando:
#   1. Metadados técnicos (table_name, column_name, data_type, nullable)
#   2. Descrições oficiais do CSV HomeCredit_columns_description
#   3. Descrições aprimoradas de business_metadata (prioridade sobre o CSV)
#   4. Descrições geradas automaticamente (para colunas sem nenhuma descrição)
#   5. Descrições a nível de tabela (table_description)
catalogo_completo = []

for row in metadados:
    col_name = row["column_name"]
    table_name = row["table_name"]

    # Descrição da tabela
    table_desc = table_descriptions.get(table_name, "")

    # Prioridade de descrição da coluna:
    # 1. business_metadata (descrições aprimoradas em português)
    # 2. csv_descriptions (descrições oficiais do dataset)
    # 3. generate_description (descrição gerada do nome)
    if col_name in business_metadata:
        description = business_metadata[col_name]["description"]
        domain = business_metadata[col_name]["domain"]
        data_classification = business_metadata[col_name]["data_classification"]
        tags = business_metadata[col_name]["tags"]
    elif (table_name, col_name) in csv_descriptions:
        description = csv_descriptions[(table_name, col_name)]
        domain = "Credit Risk"
        data_classification, auto_tags = classify_column(col_name)
        tags = auto_tags
    else:
        description = generate_description(col_name)
        domain = "Credit Risk"
        data_classification, auto_tags = classify_column(col_name)
        tags = auto_tags

    catalogo_completo.append(
        Row(
            table_name=table_name,
            table_description=table_desc,
            column_name=col_name,
            data_type=row["data_type"],
            nullable=row["nullable"],
            description=description,
            tags=tags,
            data_classification=data_classification,
            domain=domain
        )
    )

# Cria o DataFrame final do catálogo de metadados
df_catalogo = spark.createDataFrame(catalogo_completo)

# Conta cobertura de descrições
com_desc = sum(1 for r in catalogo_completo if r["description"])
sem_desc = len(catalogo_completo) - com_desc
print(f"Catálogo de metadados criado com {df_catalogo.count()} entradas")
print(f"Colunas com descrição: {com_desc} | Sem descrição: {sem_desc}")
display(df_catalogo.limit(30))

## 3. Persistência em Delta Lake

Salva o catálogo de metadados como tabela Delta em `credit_risk.bronze.metadata_catalog`.

In [0]:
# Salva o catálogo de metadados na camada Bronze como tabela Delta
# overwriteSchema=true permite evoluir o schema (ex.: adicionar table_description)
df_catalogo.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("credit_risk.bronze.metadata_catalog")

print("Tabela credit_risk.bronze.metadata_catalog criada com sucesso!")

## 4. Estatísticas do Catálogo de Metadados

In [0]:
# Estatísticas do catálogo de metadados

# Quantidade de tabelas catalogadas
qtd_tabelas = df_catalogo.select("table_name").distinct().count()
print(f"📊 Tabelas catalogadas: {qtd_tabelas}")

# Quantidade total de colunas catalogadas
qtd_colunas = df_catalogo.count()
print(f"📊 Colunas catalogadas: {qtd_colunas}")

# Quantidade de colunas por classificação de dados
print("\n📊 Colunas por classificação de dados:")
df_catalogo.groupBy("data_classification") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

# Quantidade de colunas por domínio
print("\n📊 Colunas por domínio:")
df_catalogo.groupBy("domain") \
    .count() \
    .show(truncate=False)

# Cobertura de descrições por tabela
print("\n📊 Cobertura de descrições por tabela:")
for t in sorted(set(r["table_name"] for r in catalogo_completo)):
    cols_t = [r for r in catalogo_completo if r["table_name"] == t]
    com = sum(1 for r in cols_t if r["description"])
    total = len(cols_t)
    print(f"  {t}: {com}/{total} colunas com descrição")

# Quantidade de colunas por tabela
print("\n📊 Colunas por tabela:")
df_catalogo.groupBy("table_name") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

## 5. Verificação do Catálogo Persistido

Consulta a tabela Delta `credit_risk.bronze.metadata_catalog` para confirmar a persistência e a integridade dos dados.

In [0]:
# Consulta a tabela de metadados persistida
df_metadata_salvo = spark.table("credit_risk.bronze.metadata_catalog")

# Exibe o schema da tabela
print("Schema da tabela metadata_catalog:")
df_metadata_salvo.printSchema()

# Exibe as primeiras 20 linhas
display(df_metadata_salvo.limit(20))

# Conta o total de registros persistidos
print(f"\nTotal de registros persistidos: {df_metadata_salvo.count()}")

## 6. Metadados Técnicos Estendidos, Delta Lake e Estatísticas de Tabela

Extrai automaticamente via `DESCRIBE TABLE`, `DESCRIBE DETAIL` e `DESCRIBE HISTORY`:

**Técnicos**: `catalog_name`, `schema_name`, `ordinal_position`, `precision`, `scale`, `max_length`, `partition_column`, `column_comment`
**Estatísticas**: `row_count`, `column_count`, `file_count`, `table_size_bytes`, `table_size_mb`
**Delta Lake**: `delta_version`, `last_commit_timestamp`, `last_operation`, `partition_columns`
**Contexto (auditoria)**: `user_name`, `notebook_name`, `notebook_path`, `created_at`, `ingestion_timestamp`

In [0]:
# ============================================================================
# Contexto de auditoria (auto-extraído do Spark/Databricks)
# ============================================================================
from datetime import datetime, timezone

CATALOG = "credit_risk"
SCHEMA_NAME = "bronze"
NOTEBOOK_NAME = "02_Metadados_bronze"
NOTEBOOK_PATH = "/Users/abraaojose.100@gmail.com/Projeto_classificação/Projeto_Credit_Risk/02_Metadados_bronze"
USER_NAME = spark.sql("SELECT current_user()").collect()[0][0]
EXECUTION_TS = datetime.now(timezone.utc).isoformat()

print(f"Contexto capturado:")
print(f"  Usuário: {USER_NAME}")
print(f"  Notebook: {NOTEBOOK_NAME}")
print(f"  Timestamp: {EXECUTION_TS}")

# ============================================================================
# Metadados técnicos estendidos + Delta Lake + Estatísticas de tabela
# Fontes: DESCRIBE TABLE, DESCRIBE DETAIL, DESCRIBE HISTORY, df.schema
# ============================================================================
tecnicos_ext = {}    # (table, column) → dict de metadados técnicos
tabelas_stats = {}   # table → dict de estatísticas e Delta metadata

for tabela in tabelas:
    full_table = f"{CATALOG}.{SCHEMA_NAME}.{tabela}"
    df = spark.table(full_table)

    # --- DESCRIBE TABLE: comentários de coluna ---
    desc_rows = spark.sql(f"DESCRIBE TABLE {full_table}").collect()
    col_comments = {}
    for dr in desc_rows:
        cn = dr["col_name"] or ""
        if cn and not cn.startswith("#"):
            col_comments[cn] = dr["comment"] or ""

    # --- DESCRIBE DETAIL: partição, arquivos, tamanho ---
    # Nota: DESCRIBE DETAIL pode retornar camelCase ou snake_case dependendo da versão
    detail = spark.sql(f"DESCRIBE DETAIL {full_table}").collect()[0].asDict()

    def safe_detail(d, *keys):
        """Tenta múltiplas chaves (camelCase, snake_case) e retorna o primeiro valor."""
        for k in keys:
            if k in d:
                return d[k]
        return None

    partition_cols = safe_detail(detail, "partitionColumns", "partition_columns") or []
    table_type = safe_detail(detail, "tableType", "table_type") or ""
    ds_format = safe_detail(detail, "dataSourceFormat", "data_source_format") or ""
    num_files = safe_detail(detail, "numFiles", "num_files") or 0
    size_bytes_d = safe_detail(detail, "sizeInBytes", "size_in_bytes") or 0
    created_at_val = safe_detail(detail, "createdAt", "created_at")
    last_modified_val = safe_detail(detail, "lastModified", "last_modified")

    # --- DESCRIBE HISTORY: versão Delta, última operação ---
    history = spark.sql(f"DESCRIBE HISTORY {full_table} LIMIT 1").collect()
    delta_version = int(history[0]["version"]) if history else 0
    last_commit_ts = str(history[0]["timestamp"]) if history else ""
    last_operation = history[0]["operation"] if history else ""

    # --- Estatísticas da tabela ---
    row_count = df.count()
    file_count = int(num_files)
    size_bytes = int(size_bytes_d)
    size_mb = round(size_bytes / (1024 * 1024), 2)
    col_count = len(df.columns)

    # --- Ingestion timestamp (da coluna _ingestion_timestamp) ---
    ingestion_ts = ""
    if "_ingestion_timestamp" in df.columns:
        r = df.agg({"_ingestion_timestamp": "max"}).collect()
        if r and r[0][0]:
            ingestion_ts = str(r[0][0])

    tabelas_stats[tabela] = {
        "table_type": table_type,
        "data_source_format": ds_format,
        "row_count": row_count,
        "column_count": col_count,
        "file_count": file_count,
        "table_size_bytes": size_bytes,
        "table_size_mb": size_mb,
        "delta_version": delta_version,
        "last_commit_timestamp": last_commit_ts,
        "last_operation": last_operation,
        "partition_columns": partition_cols,
        "created_at": str(created_at_val) if created_at_val else "",
        "last_modified": str(last_modified_val) if last_modified_val else "",
        "ingestion_timestamp": ingestion_ts,
    }

    # --- Metadados técnicos por coluna ---
    for idx, campo in enumerate(df.schema.fields):
        dt = campo.dataType
        precision = getattr(dt, "precision", 0) or 0
        scale = getattr(dt, "scale", 0) or 0
        max_length = getattr(dt, "length", 0) or 0

        tecnicos_ext[(tabela, campo.name)] = {
            "ordinal_position": idx,
            "precision": precision,
            "scale": scale,
            "max_length": max_length,
            "partition_column": campo.name in partition_cols,
            "column_comment": col_comments.get(campo.name, ""),
        }

    print(f"  {tabela}: {col_count} colunas | {row_count} linhas | {file_count} arquivos | {size_mb} MB | Delta v{delta_version}")

print(f"\nTotal: {len(tecnicos_ext)} colunas técnicas | {len(tabelas_stats)} tabelas")

## 7. Metadados de Qualidade de Dados

Extrai automaticamente `null_count`, `null_percentage`, `distinct_count` para cada coluna.
Usa queries batch (uma query por estatística por tabela) para eficiência — total de 16 queries para 8 tabelas em vez de 710+ queries individuais.

In [0]:
from pyspark.sql.functions import col, count, when, approx_count_distinct

# ============================================================================
# Qualidade de dados: null_count, null_percentage, distinct_count
# Queries batch: 1 para nulls + 1 para distincts = 2 queries por tabela
# ============================================================================
qualidade_lookup = {}  # (table, column) → dict

for tabela in tabelas:
    full_table = f"{CATALOG}.{SCHEMA_NAME}.{tabela}"
    df = spark.table(full_table)
    row_count = tabelas_stats[tabela]["row_count"]

    # Colunas a analisar (exclui colunas técnicas de ingestão)
    cols = [c for c in df.columns if c not in ("_ingestion_timestamp", "_source_file")]

    # --- Batch null count (1 query para todas as colunas) ---
    null_row = df.agg(*[
        count(when(col(c).isNull(), 1)).alias(c) for c in cols
    ]).collect()[0]

    # --- Batch approx distinct count (1 query para todas as colunas) ---
    distinct_row = df.agg(*[
        approx_count_distinct(c).alias(c) for c in cols
    ]).collect()[0]

    for c in cols:
        null_count = int(null_row[c] or 0)
        null_pct = round(null_count / row_count * 100, 2) if row_count > 0 else 0.0
        distinct_count = int(distinct_row[c] or 0)

        qualidade_lookup[(tabela, c)] = {
            "null_count": null_count,
            "null_percentage": null_pct,
            "distinct_count": distinct_count,
        }

    print(f"  {tabela}: qualidade calculada para {len(cols)} colunas")

print(f"\nTotal: {len(qualidade_lookup)} colunas com estatísticas de qualidade")

## 8. Metadados de Lineage e Machine Learning

**Lineage**: mapeamento fonte → destino (Volume CSV → Bronze Delta).
**ML**: flags para variável alvo (`TARGET`) e features para futuras camadas Silver/Gold.

In [0]:
# ============================================================================
# Data Lineage: Volume CSV → Bronze Delta
# Mapeamento estático baseado na origem conhecida dos dados
# ============================================================================
VOLUME_BASE = "/Volumes/credit_risk/bronze/volume/home-credit-default-risk/"
CSV_TO_TABLE = {
    "application_train.csv": "application_train",
    "application_test.csv": "application_test",
    "bureau.csv": "bureau",
    "bureau_balance.csv": "bureau_balance",
    "credit_card_balance.csv": "credit_card_balance",
    "installments_payments.csv": "installments_payments",
    "POS_CASH_balance.csv": "pos_cash_balance",
    "previous_application.csv": "previous_application",
}

# Reverse mapping: table_name → csv_file
TABLE_TO_CSV = {v: k for k, v in CSV_TO_TABLE.items()}

lineage_data = []
for csv_file, tabela in CSV_TO_TABLE.items():
    if tabela in tabelas:
        lineage_data.append(Row(
            source_catalog="",
            source_schema="",
            source_table=csv_file,
            source_file=f"{VOLUME_BASE}{csv_file}",
            target_catalog=CATALOG,
            target_schema=SCHEMA_NAME,
            target_table=tabela,
            lineage_level="bronze",
            transformation_name="CSV → Delta (Auto Loader)",
            transformation_type="ingestion",
        ))

df_lineage = spark.createDataFrame(lineage_data)
print(f"Lineage: {len(lineage_data)} mapeamentos fonte → destino")
display(df_lineage)

# ============================================================================
# Machine Learning: target_variable_flag, model_consumption_flag, feature_type
# Prepara estrutura para futuras camadas Silver e Gold
# ============================================================================
ML_FLAGS = {
    "TARGET": {"target_variable_flag": True, "feature_type": "binary_target", "feature_group": "target"},
    "SK_ID_CURR": {"target_variable_flag": False, "feature_type": "identifier", "feature_group": "key"},
    "SK_ID_BUREAU": {"target_variable_flag": False, "feature_type": "identifier", "feature_group": "key"},
    "SK_ID_PREV": {"target_variable_flag": False, "feature_type": "identifier", "feature_group": "key"},
}

ml_lookup = {}
for row in catalogo_completo:
    col_name = row["column_name"]
    if col_name in ML_FLAGS:
        ml_lookup[(row["table_name"], col_name)] = ML_FLAGS[col_name]
    else:
        is_ml_feature = "ml_feature" in row["tags"]
        ml_lookup[(row["table_name"], col_name)] = {
            "target_variable_flag": False,
            "feature_type": "numerical" if row["data_type"] in ("int", "double", "long") else "categorical",
            "feature_group": "feature" if is_ml_feature else "",
        }

print(f"\nML metadata: {len(ml_lookup)} colunas classificadas")
print(f"  Target variable: {sum(1 for v in ml_lookup.values() if v['target_variable_flag'])} coluna(s)")
print(f"  Features (ml_feature tag): {sum(1 for v in ml_lookup.values() if v['feature_group'] == 'feature')} coluna(s)")

## 9. Governança Estendida e Auditoria

**Auto-extraído**: `pii_flag`, `financial_data_flag`, `customer_data_flag`, `lgpd_flag`, `sensitivity_level`, `confidentiality_level`, `source_system`, `source_layer`, `target_layer`, `subdomain`

**Manual (estrutura preparada para preenchimento)**: `data_owner`, `data_steward`, `retention_policy`, `regulatory_requirement`, `business_definition`, `business_rule`, `critical_data_element`

In [0]:
# ============================================================================
# Governança estendida: mapeamentos, defaults e campos manuais
# ============================================================================

# Mapeamento table_name → source_file path (auto-extraído do lineage)
SOURCE_FILE_MAP = {v: f"{VOLUME_BASE}{k}" for k, v in CSV_TO_TABLE.items()}

# Sensitivity level derivado de data_classification
SENSITIVITY_MAP = {"Confidential": "High", "Internal": "Medium", "Public": "Low"}

# Defaults de governança (auto-extraídos — conhecidos do projeto)
GOV_DEFAULTS = {
    "source_system": "Home Credit Default Risk (Kaggle)",
    "source_layer": "Volume",
    "target_layer": "Bronze",
    "subdomain": "Credit Scoring",
    "domain": "Credit Risk",
}

# Campos manuais (vazios por padrao — preencher conforme governanca de negocio)
MANUAL_FIELDS = {
    "data_owner": "",
    "data_steward": "",
    "retention_policy": "",
    "regulatory_requirement": "",
    "business_definition": "",
    "business_rule": "",
}

print("Governança estendida configurada:")
print(f"  Auto-extraído: {len(GOV_DEFAULTS)} campos (source_system, source_layer, target_layer, subdomain, domain)")
print(f"  Manual: {len(MANUAL_FIELDS)} campos (data_owner, data_steward, retention_policy, etc.)")
print(f"  Auto-derived flags: pii_flag, financial_data_flag, customer_data_flag, lgpd_flag, sensitivity_level, confidentiality_level")

## 10. Catálogo Consolidado — Persistência em Múltiplas Tabelas Delta

Consolida todos os metadados em 3 tabelas Delta corporativas:

| Tabela | Escopo | Campos |
|---|---|---|
| `metadata_catalog_columns` | Coluna | 73 campos (técnico, governança, qualidade, ML, auditoria) |
| `metadata_catalog_tables` | Tabela | 32 campos (estatísticas, Delta Lake, auditoria) |
| `metadata_catalog_lineage` | Lineage | 10 campos (fonte → destino) |

In [0]:
from pyspark.sql.types import ArrayType, StringType
from pyspark.sql.functions import col as _col

# ============================================================================
# Catálogo consolidado a nível de COLUNA (73 campos)
# Combina: catalogo_completo + tecnicos_ext + qualidade_lookup + ml_lookup + governance
# ============================================================================
catalogo_final = []

for row in catalogo_completo:
    table_name = row["table_name"]
    col_name = row["column_name"]
    tags = row["tags"]

    # Lookups de metadados extraídos nas células anteriores
    tec = tecnicos_ext.get((table_name, col_name), {})
    qual = qualidade_lookup.get((table_name, col_name), {})
    ml = ml_lookup.get((table_name, col_name), {})
    stats = tabelas_stats.get(table_name, {})

    # Flags de governança derivadas automaticamente das tags
    pii_flag = "pii" in tags
    financial_flag = "financial" in tags
    customer_flag = "customer" in tags
    lgpd_flag = pii_flag

    sensitivity = SENSITIVITY_MAP.get(row["data_classification"], "Medium")
    source_file = SOURCE_FILE_MAP.get(table_name, "")
    source_csv = TABLE_TO_CSV.get(table_name, "")
    critical = col_name == "TARGET"

    record = {
        # --- Metadados Técnicos (16) ---
        "catalog_name": CATALOG,
        "schema_name": SCHEMA_NAME,
        "table_name": table_name,
        "table_description": row["table_description"],
        "table_comment": "",
        "column_name": col_name,
        "ordinal_position": tec.get("ordinal_position", 0),
        "data_type": row["data_type"],
        "nullable": row["nullable"],
        "precision": tec.get("precision", 0),
        "scale": tec.get("scale", 0),
        "max_length": tec.get("max_length", 0),
        "partition_column": tec.get("partition_column", False),
        "generated_column": False,
        "default_value": "",
        "column_comment": tec.get("column_comment", ""),
        # --- Metadados de Governança (22) ---
        "description": row["description"],
        "business_definition": MANUAL_FIELDS["business_definition"],
        "business_rule": MANUAL_FIELDS["business_rule"],
        "domain": GOV_DEFAULTS["domain"],
        "subdomain": GOV_DEFAULTS["subdomain"],
        "data_owner": MANUAL_FIELDS["data_owner"],
        "data_steward": MANUAL_FIELDS["data_steward"],
        "source_system": GOV_DEFAULTS["source_system"],
        "source_table": source_csv,
        "source_file": source_file,
        "source_layer": GOV_DEFAULTS["source_layer"],
        "target_layer": GOV_DEFAULTS["target_layer"],
        "data_classification": row["data_classification"],
        "sensitivity_level": sensitivity,
        "confidentiality_level": row["data_classification"],
        "retention_policy": MANUAL_FIELDS["retention_policy"],
        "regulatory_requirement": MANUAL_FIELDS["regulatory_requirement"],
        "lgpd_flag": lgpd_flag,
        "pii_flag": pii_flag,
        "financial_data_flag": financial_flag,
        "customer_data_flag": customer_flag,
        "critical_data_element": critical,
        # --- Tags (1) ---
        "tags": tags,
        # --- Metadados de Qualidade (10) ---
        "null_count": qual.get("null_count", 0),
        "null_percentage": qual.get("null_percentage", 0.0),
        "distinct_count": qual.get("distinct_count", 0),
        "duplicate_count": 0,
        "duplicate_percentage": 0.0,
        "min_value": "",
        "max_value": "",
        "avg_value": "",
        "data_quality_score": 0.0,
        "quality_status": "pending",
        # --- Metadados ML (8) ---
        "feature_name": col_name if ml.get("feature_group") else "",
        "feature_description": row["description"] if ml.get("feature_group") else "",
        "feature_type": ml.get("feature_type", ""),
        "feature_owner": "",
        "feature_group": ml.get("feature_group", ""),
        "feature_importance": 0.0,
        "model_consumption_flag": "ml_feature" in tags,
        "target_variable_flag": ml.get("target_variable_flag", False),
        # --- Metadados de Auditoria (16) ---
        "ingestion_timestamp": stats.get("ingestion_timestamp", ""),
        "processing_timestamp": "",
        "load_timestamp": "",
        "created_at": EXECUTION_TS,
        "updated_at": EXECUTION_TS,
        "batch_id": "",
        "execution_id": "",
        "pipeline_id": "",
        "notebook_name": NOTEBOOK_NAME,
        "notebook_path": NOTEBOOK_PATH,
        "user_name": USER_NAME,
        "workspace_name": "",
        "cluster_id": "",
        "job_id": "",
        "run_id": "",
        "execution_status": "completed",
    }
    catalogo_final.append(Row(**record))

df_catalog_columns = spark.createDataFrame(catalogo_final)

# ============================================================================
# Catálogo a nível de TABELA (32 campos)
# ============================================================================
catalogo_tabelas = []
for tabela in tabelas:
    stats = tabelas_stats.get(tabela, {})
    size_mb = stats.get("table_size_mb", 0.0)
    row_cnt = stats.get("row_count", 1)
    size_bytes = stats.get("table_size_bytes", 0)

    catalogo_tabelas.append(Row(**{
        "catalog_name": CATALOG,
        "schema_name": SCHEMA_NAME,
        "table_name": tabela,
        "table_description": table_descriptions.get(tabela, ""),
        "table_comment": "",
        "table_type": stats.get("table_type", ""),
        "data_source_format": stats.get("data_source_format", ""),
        "row_count": row_cnt,
        "column_count": stats.get("column_count", 0),
        "file_count": stats.get("file_count", 0),
        "table_size_bytes": size_bytes,
        "table_size_mb": size_mb,
        "table_size_gb": round(size_mb / 1024, 4),
        "avg_record_size": round(size_bytes / max(row_cnt, 1), 2),
        "delta_table": True,
        "delta_version": stats.get("delta_version", 0),
        "last_commit_timestamp": stats.get("last_commit_timestamp", ""),
        "last_operation": stats.get("last_operation", ""),
        "partition_columns": ",".join(stats.get("partition_columns", [])),
        "table_created_at": stats.get("created_at", ""),
        "last_modified": stats.get("last_modified", ""),
        "domain": GOV_DEFAULTS["domain"],
        "subdomain": GOV_DEFAULTS["subdomain"],
        "data_owner": MANUAL_FIELDS["data_owner"],
        "data_steward": MANUAL_FIELDS["data_steward"],
        "data_classification": "Confidential",
        "retention_policy": MANUAL_FIELDS["retention_policy"],
        "ingestion_timestamp": stats.get("ingestion_timestamp", ""),
        "notebook_name": NOTEBOOK_NAME,
        "notebook_path": NOTEBOOK_PATH,
        "user_name": USER_NAME,
        "created_at": EXECUTION_TS,
    }))

df_catalog_tables = spark.createDataFrame(catalogo_tabelas)

# partition_columns armazenado como string (comma-separated) para compatibilidade

# ============================================================================
# Persistência em 3 tabelas Delta
# ============================================================================
df_catalog_columns.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("credit_risk.bronze.metadata_catalog_columns")

df_catalog_tables.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("credit_risk.bronze.metadata_catalog_tables")

df_lineage.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("credit_risk.bronze.metadata_catalog_lineage")

print("3 tabelas Delta criadas com sucesso:")
print("  • credit_risk.bronze.metadata_catalog_columns")
print("  • credit_risk.bronze.metadata_catalog_tables")
print("  • credit_risk.bronze.metadata_catalog_lineage")
print(f"\n  Colunas: {df_catalog_columns.count()} registros, {len(df_catalog_columns.columns)} campos")
print(f"  Tabelas: {df_catalog_tables.count()} registros, {len(df_catalog_tables.columns)} campos")
print(f"  Lineage: {df_lineage.count()} registros, {len(df_lineage.columns)} campos")

## 11. Estatísticas Finais e Verificação

Exibe estatísticas consolidadas de todas as tabelas de metadados e classifica campos como automáticos vs manuais.

In [0]:
# ============================================================================
# Estatísticas finais do catálogo corporativo de metadados
# ============================================================================

print("=" * 70)
print("CATÁLOGO CORPORATIVO DE METADADOS — CREDIT RISK BRONZE")
print("=" * 70)

# --- Tabela de colunas ---
print(f"\n📊 metadata_catalog_columns:")
print(f"   Registros: {df_catalog_columns.count()}")
print(f"   Campos: {len(df_catalog_columns.columns)}")
print(f"   Tabelas cobertas: {df_catalog_columns.select('table_name').distinct().count()}")

# Cobertura de descrições
com_desc = df_catalog_columns.filter("description != ''").count()
total = df_catalog_columns.count()
print(f"   Cobertura de descrições: {com_desc}/{total} ({round(com_desc/total*100, 1)}%)")

# Cobertura de qualidade
print(f"   Qualidade calculada: {len(qualidade_lookup)} colunas")

# --- Flags de governança ---
print(f"\n📊 Flags de governança:")
print(f"   PII: {df_catalog_columns.filter('pii_flag = true').count()} colunas")
print(f"   Financial: {df_catalog_columns.filter('financial_data_flag = true').count()} colunas")
print(f"   Customer: {df_catalog_columns.filter('customer_data_flag = true').count()} colunas")
print(f"   LGPD: {df_catalog_columns.filter('lgpd_flag = true').count()} colunas")
print(f"   Target variable: {df_catalog_columns.filter('target_variable_flag = true').count()} coluna(s)")
print(f"   Critical data element: {df_catalog_columns.filter('critical_data_element = true').count()} coluna(s)")

# --- Tabela de tabelas ---
print(f"\n📊 metadata_catalog_tables:")
df_catalog_tables.select(
    "table_name", "row_count", "column_count", "file_count", "table_size_mb", "delta_version"
).show(truncate=False)

# --- Lineage ---
print(f"\n📊 metadata_catalog_lineage:")
df_lineage.show(truncate=False)

# --- Distribuição por classificação ---
print(f"\n📊 Distribuição por classificação de dados:")
df_catalog_columns.groupBy("data_classification").count().orderBy("count", ascending=False).show(truncate=False)

# --- Sensitivity level ---
print(f"📊 Distribuição por sensitivity level:")
df_catalog_columns.groupBy("sensitivity_level").count().orderBy("count", ascending=False).show(truncate=False)

# --- Colunas por tabela ---
print(f"📊 Colunas por tabela:")
df_catalog_columns.groupBy("table_name").count().orderBy("count", ascending=False).show(truncate=False)

# --- Resumo: auto vs manual ---
print(f"\n📋 Origem dos metadados:")
print(f"   ✅ AUTO-EXTRAÍDO (Spark/UC/Delta Lake):")
print(f"      catalog_name, schema_name, table_name, column_name, ordinal_position,")
print(f"      data_type, nullable, precision, scale, max_length, partition_column,")
print(f"      column_comment, row_count, file_count, table_size_bytes, delta_version,")
print(f"      last_commit_timestamp, last_operation, partition_columns,")
print(f"      null_count, null_percentage, distinct_count,")
print(f"      pii_flag, financial_data_flag, customer_data_flag, lgpd_flag,")
print(f"      sensitivity_level, confidentiality_level, source_system, source_layer,")
print(f"      target_layer, subdomain, target_variable_flag, model_consumption_flag,")
print(f"      feature_type, feature_group, ingestion_timestamp,")
print(f"      notebook_name, notebook_path, user_name, created_at, execution_status")
print(f"   📝 MANUAL (preencher com governança de negócio):")
print(f"      data_owner, data_steward, retention_policy, regulatory_requirement,")
print(f"      business_definition, business_rule, critical_data_element,")
print(f"      default_value, generated_column, table_comment,")
print(f"      feature_owner, feature_importance, data_quality_score, quality_status,")
print(f"      min_value, max_value, avg_value, duplicate_count, duplicate_percentage,")
print(f"      batch_id, execution_id, pipeline_id, job_id, run_id,")
print(f"      workspace_name, cluster_id, processing_timestamp, load_timestamp")